In [2]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [4]:
import json
import random
import torch
import gc
from tqdm.auto import tqdm
from PIL import Image
from unsloth import FastVisionModel

# --- 1. CẤU HÌNH ĐƯỜNG DẪN ---
# Thay bằng đường dẫn thực tế của bạn
TRAIN_JSON_PATH = "/kaggle/input/datasets/spixalo/trafic/traffic-signs-json/train.json"
IMAGE_DIR = "/kaggle/input/datasets/maitam/vietnamese-traffic-signs/archive/images"
MODEL_NAME = "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit"
ADAPTER_PATH = "/kaggle/input/datasets/spixalo/checkpoint-750/checkpoint-750"

# --- 2. NẠP VÀ TRỘN 1000 DỮ LIỆU ---
with open(TRAIN_JSON_PATH, 'r', encoding='utf-8') as f:
    full_train_data = json.load(f)

# Trộn lên và lấy 1000 mẫu ngẫu nhiên
random.seed(42) # Set seed để lần sau chạy lại vẫn ra đúng 1000 mẫu này
sampled_data = random.sample(full_train_data, 1000)
print(f"Đã bốc thăm {len(sampled_data)} mẫu từ tập Train.")

# --- 3. LOAD MÔ HÌNH B2 ---
print("Đang nạp mô hình B2...")
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME, load_in_4bit=True,
)
model.load_adapter(ADAPTER_PATH)
FastVisionModel.for_inference(model)

# --- 4. CHẠY INFERENCE ---
inference_results = []

for item in tqdm(sampled_data, desc="Đang suy luận 1000 mẫu"):
    img_path = f"{IMAGE_DIR}/{item['image_name']}"
    question = item["question"]
    ground_truth = item["answer"][0]
    
    try:
        img = Image.open(img_path).convert("RGB")
        prompt = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": question}]}]
        
        inputs = tokenizer.apply_chat_template(prompt, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=15, use_cache=True, temperature=0.1, do_sample=False)
            
        prediction = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
        
        inference_results.append({
            "image_path": img_path,
            "question": question,
            "ground_truth": ground_truth,
            "prediction": prediction
        })
    except Exception as e:
        pass # Bỏ qua ảnh lỗi để không đứt gánh giữa đường

# --- 5. LƯU NHÁP VÀ "TẨY NÃO" VRAM ---
with open("/kaggle/working/temp_inference.json", "w", encoding="utf-8") as f:
    json.dump(inference_results, f, ensure_ascii=False, indent=2)

print("Đã lưu nháp. Đang tiến hành xóa mô hình để giải phóng VRAM...")

# Hủy diệt model khỏi bộ nhớ
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()

print("VRAM đã sạch sẽ! Sẵn sàng cho bước chấm điểm.")

Đã bốc thăm 1000 mẫu từ tập Train.
Đang nạp mô hình B2...
==((====))==  Unsloth 2026.5.2: Fast Qwen2_Vl patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

Đang suy luận 1000 mẫu:   0%|          | 0/1000 [00:00<?, ?it/s]

Đã lưu nháp. Đang tiến hành xóa mô hình để giải phóng VRAM...
VRAM đã sạch sẽ! Sẵn sàng cho bước chấm điểm.


In [6]:
!pip install evaluate rouge_score nltk bert_score

import json
import evaluate

# --- 1. LOAD DATA NHÁP ---
with open("/kaggle/working/temp_inference.json", "r", encoding="utf-8") as f:
    inference_results = json.load(f)

predictions = [item["prediction"] for item in inference_results]
references = [item["ground_truth"] for item in inference_results]

# --- 2. CHẤM ĐIỂM BERTSCORE ---
print("Đang nạp giám khảo BERTScore (RoBERTa)...")
bertscore = evaluate.load("bertscore")

print("Đang chấm điểm ngữ nghĩa cho hàng ngàn câu (sẽ tốn vài phút)...")
bert_results = bertscore.compute(predictions=predictions, references=references, lang="vi")

# Gắn điểm F1 vào lại từng mẫu dữ liệu
for i, item in enumerate(inference_results):
    item["bertscore"] = bert_results["f1"][i]

# --- 3. LỌC 200 CÂU TỆ NHẤT VÀ FORMAT CHUẨN DPO ---
# Sắp xếp tăng dần theo điểm BERTScore (câu ngu nhất nằm đầu)
inference_results.sort(key=lambda x: x["bertscore"])
worst_200 = inference_results[:200]

dpo_dataset = []
for item in worst_200:
    # Đóng gói chuẩn format DPO cho mô hình Multimodal
    dpo_sample = {
        "prompt": [
            {"role": "user", "content": [
                {"type": "image", "image": item["image_path"]}, 
                {"type": "text", "text": item["question"]}
            ]}
        ],
        "chosen": [
            {"role": "assistant", "content": [{"type": "text", "text": item["ground_truth"]}]}
        ],
        "rejected": [
            {"role": "assistant", "content": [{"type": "text", "text": item["prediction"]}]}
        ],
        # Lưu lại điểm để bạn đối chiếu lúc kiểm duyệt
        "debug_info": f"BERTScore: {item['bertscore']:.4f}" 
    }
    dpo_dataset.append(dpo_sample)

# --- 4. LƯU THÀNH QUẢ ---
OUTPUT_FILE = "/kaggle/working/dpo_preference_data_200.json"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(dpo_dataset, f, ensure_ascii=False, indent=2)

print(f"\n🎉 XONG! Đã lọc thành công 200 câu tệ nhất.")
print(f"File DPO đã được lưu tại: {OUTPUT_FILE}")
print("Nhiệm vụ của bạn: Mở file json này lên, kiểm tra lại phần 'rejected' xem nó có đủ 'sai trái' chưa nhé!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.1 MB/s eta 0:00:00
Đang nạp giám khảo BERTScore (RoBERTa)...


Đang chấm điểm ngữ nghĩa cho hàng ngàn câu (sẽ tốn vài phút)...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]


🎉 XONG! Đã lọc thành công 200 câu tệ nhất.
File DPO đã được lưu tại: /kaggle/working/dpo_preference_data_200.json
Nhiệm vụ của bạn: Mở file json này lên, kiểm tra lại phần 'rejected' xem nó có đủ 'sai trái' chưa nhé!
